# Hybrid Analytics System Dashboard
MongoDB + Cassandra Analytics

In [ ]:
%pip install plotly pandas

In [9]:
import pandas as pd
import plotly.express as px
from pymongo import MongoClient
from cassandra.cluster import Cluster

In [10]:
# Connections
mongo = MongoClient('mongodb://localhost:27017')
mongo_db = mongo.hybrid_analytics
products = mongo_db.products

cluster = Cluster(['localhost'])
session = cluster.connect('hybrid_analytics')

print('Connected Successfully')

Connected Successfully


## Products by Category

In [21]:
pipeline = [{'$group': {'_id': '$category', 'count': {'$sum': 1}}}]
category_df = pd.DataFrame(list(products.aggregate(pipeline)))
category_df.columns = ['category','count']
fig = px.bar(
    category_df,
    x='category',
    y='count',
    color='category',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Category'
)
fig.show()

## Products by Brand

In [22]:
pipeline = [{'$group': {'_id': '$brand', 'count': {'$sum': 1}}}]
brand_df = pd.DataFrame(list(products.aggregate(pipeline)))
brand_df.columns = ['brand','count']
fig = px.bar(
    brand_df.sort_values('count'),
    x='count',
    y='brand',
    color='brand',
    orientation='h',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Brand'
)
fig.update_layout(showlegend=False)
fig.show()

## Price Distribution

In [30]:
fig = px.histogram(prices, x='price', nbins=30, title='Price Distribution Histogram')
fig.show()


## Average Price by Category

In [32]:
fig = px.bar(
    avg_df,
    x='category',
    y='avg_price',
    color='category',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Average Price by Category'
)
fig.update_layout(showlegend=False)
fig.show()

## Cassandra Metrics

In [33]:
rows = session.execute('SELECT * FROM product_metrics')
metrics_df = pd.DataFrame(list(rows))
metrics_df.head()

,product_id,event_type,count
0,23,buy,8
1,23,favorite,33
2,23,view,155
3,114,buy,13
4,114,favorite,38


## Top Viewed Products

In [43]:
views_products = views_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)

views_products['label'] = views_products['title'] + ' (' + views_products['brand'] + ')'

fig = px.pie(
    views_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Viewed Products'
)
fig.show()

## Top Favorited Products

In [42]:
fav_df = metrics_df[metrics_df['event_type'] == 'favorite'].sort_values('count', ascending=False).head(10)
fav_products = fav_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
fav_products['label'] = fav_products['title'] + ' (' + fav_products['brand'] + ')'

fig = px.pie(
    fav_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Favorited Products'
)
fig.show()

## Top Purchased Products

In [41]:
buy_df = metrics_df[metrics_df['event_type'] == 'buy'].sort_values('count', ascending=False).head(10)
buy_products = buy_df.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
buy_products['label'] = buy_products['title'] + ' (' + buy_products['brand'] + ')'

fig = px.pie(
    buy_products,
    names='label',
    values='count',
    color='label',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Top Purchased Products'
)
fig.show()

## Product Engagement Score

In [40]:
fig = px.bar(
    brand_df,
    x='brand',
    y='count',
    color='brand',
    color_discrete_sequence=px.colors.qualitative.Plotly,
    title='Products by Brand'
)
fig.update_layout(showlegend=False)
fig.show()

In [49]:
query = "SELECT product_id, event_type, count FROM product_metrics WHERE product_id = 2"

rows = session.execute(query)

metrics_df = pd.DataFrame(list(rows))

metrics_pivot = (
    metrics_df
    .pivot(index='product_id', columns='event_type', values='count')
    .reset_index()
    .fillna(0)
)
metrics_pivot.columns.name = None
metrics_pivot = metrics_pivot.astype({'buy': int, 'favorite': int, 'view': int})

metrics_pivot = metrics_pivot.merge(
    product_df[['id', 'title', 'brand']],
    left_on='product_id',
    right_on='id',
    how='left'
)
metrics_pivot.head()

,product_id,buy,favorite,view,id,title,brand
0,2,6,33,170,2,Total motivating moratorium,LG
